In [ ]:
import pandas as pd
import datetime

# 读取 CSV 文件
df = pd.read_csv('./wenjian/S1.csv')

# 显示转换后的数据框
print(df)


In [6]:
# 将日期列转换为 datetime 格式
df['charttime'] = pd.to_datetime(df['charttime'])
df['starttime'] = pd.to_datetime(df['starttime'])

# 比较 A 列中的日期时间是否大于 B 列中的日期时间
df['charttime_greater_than_starttime'] = df['charttime'] > df['starttime']

# 显示包含比较结果的数据框
print(df)


      subject_id   hadm_id   stay_id  caregiver_id           charttime  \
0       10035168  25449821  39978210           NaN 2145-12-26 17:29:00   
1       10035168  25449821  39978210           NaN 2145-12-26 17:29:00   
2       10035168  25449821  39978210           NaN 2145-12-26 17:29:00   
3       10035168  25449821  39978210           NaN 2145-12-26 17:29:00   
4       10035168  25449821  39978210           NaN 2145-12-26 17:29:00   
...          ...       ...       ...           ...                 ...   
5080    19970491  25338284  31703881           NaN 2129-07-19 06:00:00   
5081    19970491  25338284  31703881           NaN 2129-07-19 19:26:00   
5082    19970491  25338284  31703881           NaN 2129-07-19 19:26:00   
5083    19970491  25338284  31703881           NaN 2129-07-19 19:26:00   
5084    19985293  21731208  34896989           NaN 2184-08-20 13:41:00   

             storetime  itemid  value  valuenum valueuom  ...  \
0     2145/12/26 19:22  227449    2.7       2.

In [8]:
# 筛选出比较结果为 True 的行
filtered_df = df[df['charttime_greater_than_starttime']]

# 保存筛选后的数据到新的 CSV 文件
filtered_df.to_csv('./wenjian/S2.csv', index=False)

print("包含比较结果为 True 的行已保存到 S2.csv 文件中。")


包含比较结果为 True 的行已保存到 S2.csv 文件中。


In [4]:
import pandas as pd

# 读取S2.csv文件
df = pd.read_csv('S2.csv')

# 将'starttime'和'charttime'列转换为日期时间格式
df['starttime'] = pd.to_datetime(df['starttime'])
df['charttime'] = pd.to_datetime(df['charttime'])

# 计算时间差并保留小数点后一位
df['chalie'] = round((df['charttime'] - df['starttime']).dt.total_seconds() / 3600, 1)

# 保存修改后的数据到新的CSV文件
df.to_csv('S2_with_chalie_hours_decimal.csv', index=False)


In [46]:
# 4. 在满足时间条件前提下选取最接近浓度测试时间的剂量输注数据
# 读取 S2.csv 文件
df = pd.read_csv('./wenjian/S2.csv')

# 将 starttime 列转换为 datetime 格式
df['starttime'] = pd.to_datetime(df['starttime'])

# 根据 charttime 列进行分组，并选择每组中 starttime 列数据最大的一行
max_starttime_rows = df.loc[df.groupby('charttime')['starttime'].idxmax()]

# 保存数据到新的 CSV 文件
max_starttime_rows.to_csv('S3.csv', index=False)

# 打印数据
print("在 charttime 列数据相同的前提下，选择 starttime 列数据最大的一行：")
print(max_starttime_rows)


在 charttime 列数据相同的前提下，选择 starttime 列数据最大的一行：
      subject_id   hadm_id   stay_id  caregiver_id            charttime  \
2000    16504173  25082363  35859388           NaN  2110-06-20 09:17:00   
2006    16504173  25082363  35859388           NaN  2110-06-20 12:20:00   
1870    15795343  26631591  32303877           NaN  2112-03-12 22:47:00   
2706    19122448  24507586  36404824           NaN  2112-12-13 04:07:00   
2707    19122448  24507586  36404824           NaN  2112-12-15 02:19:00   
...          ...       ...       ...           ...                  ...   
648     11972365  21606206  31850358           NaN  2198-03-25 11:13:00   
649     11972365  21606206  31850358           NaN  2198-03-26 05:17:00   
1089    14065397  25219971  39439699           NaN  2199-11-20 15:11:00   
1093    14065397  25219971  39439699           NaN  2199-11-20 18:03:00   
2346    17417573  24710404  36358936           NaN  2208-08-12 13:55:00   

             storetime  itemid  value  valuenum valueu

In [1]:
import pandas as pd

# 读取labs.csv和S3.csv文件
labs_df = pd.read_csv('./wenjian/labs.csv')
s3_df = pd.read_csv('./wenjian/S3.csv')

# 创建一个字典来收集新列的数据
new_columns = {}

# 遍历labs_df中的每个唯一的itemid和对应的valueuom
for itemid, group in labs_df.groupby('itemid'):
    # 假设每个itemid对应一个固定的valueuom值，取该组的第一个valueuom作为代表
    valueuom = group['valueuom'].iloc[0] if not group['valueuom'].isnull().all() else 'default_uom'

    # 创建新列的名称和数据
    column_name = f'{itemid}'
    column_data = [None] * len(s3_df)  # 初始化与s3_df长度相同的数据列表

    # 将新列的名称和数据添加到字典中
    new_columns[column_name] = column_data

    # 同时为valueuom创建列（如果valueuom不随记录变化）
    valueuom_column_name = f'valueuom_{itemid}'
    valueuom_column_data = [valueuom] * len(s3_df)  # 所有行都是相同的valueuom
    new_columns[valueuom_column_name] = valueuom_column_data

# 将新列一次性添加到s3_df中
for column_name, column_data in new_columns.items():
    s3_df[column_name] = column_data

# 保存处理后的DataFrame为新的CSV文件
s3_df.to_csv('S3_modified.csv', index=False)

###############################################################################################################

s3_modified_path = 'S3_modified.csv'
labs_path = './wenjian/labs.csv'

s3_modified_df = pd.read_csv(s3_modified_path)
labs_df = pd.read_csv(labs_path)

# 时间格式处理
s3_modified_df['charttime'] = pd.to_datetime(s3_modified_df['charttime'], format='%Y/%m/%d %H:%M')
labs_df['charttime'] = pd.to_datetime(labs_df['charttime'], format='%Y/%m/%d %H:%M')

# 构建一个函数来查找并填充最接近的记录
def fill_closest_values(s3_df, labs_df):
    # 准备一个空的 DataFrame 用于存放结果
    result_df = s3_df.copy()
    # 从第三列开始的所有奇数列是我们需要填充的列
    columns_to_fill = s3_df.columns[2::2]

    for col in columns_to_fill:
        itemid = int(col)
        # 对每个需要填充的列，使用 merge_asof 来寻找最近的记录
        # 需要注意的是，merge_asof 要求两个 DataFrame 都是按时间排序的
        merged_df = pd.merge_asof(s3_df.sort_values('charttime'),
                                  labs_df[labs_df['itemid'] == itemid].sort_values('charttime'),
                                  on='charttime',
                                  by='stay_id',
                                  direction='nearest')
        # 将找到的值填充到结果 DataFrame 中
        result_df[col] = merged_df['value']

    return result_df

# 调用函数进行填充
filled_df = fill_closest_values(s3_modified_df, labs_df)

# 完成数据填充后，保存结果到一个新的 CSV 文件
filled_df.to_csv('./wenjian/S4.csv', index=False)



C:\Windows\Temp\ipykernel_11124\1692038882.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  s3_df[column_name] = column_data
C:\Windows\Temp\ipykernel_11124\1692038882.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  s3_df[column_name] = column_data
C:\Windows\Temp\ipykernel_11124\1692038882.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a 

In [14]:
import pandas as pd

# 读取 S4.csv 和 GA.csv 文件
df_s4 = pd.read_csv('./wenjian/S4.csv')
df_ga = pd.read_csv('./wenjian/GA.csv')

# 根据 subject_id 列将两个数据集连接
merged_df = pd.merge(df_s4, df_ga, on='subject_id', how='left')

# 将连接后的数据写入新的 CSV 文件
merged_df.to_csv('S5.csv', index=False)

In [8]:
import pandas as pd

# 读取 BMH.CSV 文件
df = pd.read_csv('./wenjian/BMH.csv')

# 将 result_value 列转换为数值类型
df['result_value'] = pd.to_numeric(df['result_value'], errors='coerce')

# 透视表操作
pivot_df = df.pivot_table(index=['subject_id', 'chartdate', 'seq_num'], columns='result_name', values='result_value', aggfunc='first').reset_index()

# 重命名列名
pivot_df.rename(columns={'BMI (kg/m2)': 'BMI (kg/m2)', 'Height (Inches)': 'Height (Inches)', 'Blood Pressure': 'Blood Pressure'}, inplace=True)

# 保存调整后的数据到新的 CSV 文件
pivot_df.to_csv('BMH1.csv', index=False)

# 打印数据
print("调整后的数据为：")
print(pivot_df)

调整后的数据为：
result_name  subject_id   chartdate  seq_num  BMI (kg/m2)  Height (Inches)  \
0              10035168  2143/12/21        1         22.2              NaN   
1              10035168  2143/12/24        1         22.6              NaN   
2              10035168  2143/12/26        1         22.2              NaN   
3              10035168  2143/12/28        1         22.1              NaN   
4              10035168  2143/12/31        1         22.2              NaN   
...                 ...         ...      ...          ...              ...   
3628           19985293    2184/7/3        1         24.6             60.0   
3629           19985293   2184/8/13        1         27.3              NaN   
3630           19985293   2184/8/18        1         26.3              NaN   
3631           19985293    2184/8/3        1         24.1              NaN   
3632           19985293    2184/8/5        1         26.8             56.0   

result_name  Weight  Weight (Lbs)  
0               Na

In [11]:
import pandas as pd

# 读取 S5.csv 文件
df_s5 = pd.read_csv('./wenjian/S5.csv')

# 读取 BMH1.csv 文件
df_s51 = pd.read_csv('./wenjian/BMH1.csv')

# 将 S5.csv 文件中的 charttime 列转换为日期时间类型
df_s5['charttime'] = pd.to_datetime(df_s5['charttime'])

# 将 BMH1.csv 文件中的 chartdate 列转换为日期时间类型
df_s51['chartdate'] = pd.to_datetime(df_s51['chartdate'])

# 合并数据
merged_df = pd.merge_asof(df_s5.sort_values('charttime'), df_s51.sort_values('chartdate'), by='subject_id', left_on='charttime', right_on='chartdate', direction='backward')

# 保存合并后的数据到新的 CSV 文件
merged_df.to_csv('S6.csv', index=False)

# 打印数据
print("合并后的数据为：")
print(merged_df)


合并后的数据为：
     subject_id   hadm_id   stay_id           charttime  220224  \
0      16504173  25082363  35859388 2110-06-20 09:17:00    91.0   
1      16504173  25082363  35859388 2110-06-20 12:20:00    91.0   
2      15795343  26631591  32303877 2112-03-12 22:47:00   210.0   
3      19122448  24507586  36404824 2112-12-13 04:07:00    65.0   
4      19122448  24507586  36404824 2112-12-15 02:19:00    65.0   
..          ...       ...       ...                 ...     ...   
748    11972365  21606206  31850358 2198-03-25 11:13:00     NaN   
749    11972365  21606206  31850358 2198-03-26 05:17:00     NaN   
750    14065397  25219971  39439699 2199-11-20 15:11:00    94.0   
751    14065397  25219971  39439699 2199-11-20 18:03:00    94.0   
752    17417573  24710404  36358936 2208-08-12 13:55:00    92.0   

    valueuom_220224  220227 valueuom_220227  220228 valueuom_220228  ...  \
0              mmHg    97.0               %     8.4            g/dl  ...   
1              mmHg    97.0       

In [7]:
import pandas as pd

# 读取 HS.CSV 文件
df = pd.read_csv('./wenjian/HS.csv')

# 将 result_value 列转换为数值类型
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# 保留 itemid=226730 的数据
df = df[df['itemid'] == 226730]

# 透视表操作
pivot_df = df.pivot_table(index=['stay_id', 'charttime'], columns='itemid', values='value', aggfunc='first').reset_index()

# 重命名列名
pivot_df.rename(columns={'226730': '226730'}, inplace=True)

# 保存调整后的数据到新的 CSV 文件
pivot_df.to_csv('HS1.csv', index=False)

# 打印数据
print("调整后的数据为：")
print(pivot_df)

调整后的数据为：
itemid   stay_id         charttime  226730
0       30100021   2128/6/20 21:32   157.0
1       30103961    2114/3/7 14:45   157.0
2       30170996   2138/1/19 10:39   150.0
3       30252483   2177/7/13 23:43   168.0
4       30311354     2163/6/7 4:15   157.0
..           ...               ...     ...
207     39762626   2128/5/12 17:19   163.0
208     39785216   2195/10/13 9:50   173.0
209     39817695   2146/10/9 14:51   175.0
210     39965206  2166/12/17 21:39   183.0
211     39978210  2145/12/24 23:19   170.0

[212 rows x 3 columns]


In [20]:
import pandas as pd

# 读取 HS1.csv 文件
hs_df = pd.read_csv('./wenjian/HS1.csv')
hs_df['charttime1'] = pd.to_datetime(hs_df['charttime1'])  # 将日期时间列转换为日期时间类型

# 读取 S6.csv 文件
s6_df = pd.read_csv('./wenjian/S6.csv')
s6_df['charttime'] = pd.to_datetime(s6_df['charttime'])  # 将日期时间列转换为日期时间类型

# 使用 merge_asof 函数将数据填充到 S6.csv 文件后面
merged_df = pd.merge_asof(s6_df.sort_values('charttime'),
                          hs_df.sort_values('charttime1'),
                          left_on='charttime',
                          right_on='charttime1',
                          by='stay_id',
                          direction='nearest')

# 保存填充后的数据到新的 CSV 文件
merged_df.to_csv('S7.csv', index=False)

# 打印填充后的数据
print("填充后的数据为：")
print(merged_df)


填充后的数据为：
     subject_id   hadm_id   stay_id           charttime  220224  \
0      16504173  25082363  35859388 2110-06-20 09:17:00    91.0   
1      16504173  25082363  35859388 2110-06-20 12:20:00    91.0   
2      15795343  26631591  32303877 2112-03-12 22:47:00   210.0   
3      19122448  24507586  36404824 2112-12-13 04:07:00    65.0   
4      19122448  24507586  36404824 2112-12-15 02:19:00    65.0   
..          ...       ...       ...                 ...     ...   
748    11972365  21606206  31850358 2198-03-25 11:13:00     NaN   
749    11972365  21606206  31850358 2198-03-26 05:17:00     NaN   
750    14065397  25219971  39439699 2199-11-20 15:11:00    94.0   
751    14065397  25219971  39439699 2199-11-20 18:03:00    94.0   
752    17417573  24710404  36358936 2208-08-12 13:55:00    92.0   

    valueuom_220224  220227 valueuom_220227  220228 valueuom_220228  ...  \
0              mmHg    97.0               %     8.4            g/dl  ...   
1              mmHg    97.0       

In [2]:
import pandas as pd

# 读取 ethnic.csv 文件和 S7.csv 文件
df_ethnic = pd.read_csv('./wenjian/ethnic.csv')
df_s7 = pd.read_csv('./wenjian/S7.csv')

# 将 race 列数据根据 subject_id 填充到 S7.csv 后面
df_merged = pd.merge(df_s7, df_ethnic[['subject_id', 'race']], on='subject_id', how='left')

# 将合并后的数据保存为新的 CSV 文件
df_merged.to_csv('S8.csv', index=False)

# 打印填充后的数据
print("填充后的数据为：")
print(df_merged)

填充后的数据为：
     subject_id   hadm_id   stay_id            charttime  220224  \
0      16504173  25082363  35859388  2110-06-20 09:17:00    91.0   
1      16504173  25082363  35859388  2110-06-20 12:20:00    91.0   
2      15795343  26631591  32303877  2112-03-12 22:47:00   210.0   
3      19122448  24507586  36404824  2112-12-13 04:07:00    65.0   
4      19122448  24507586  36404824  2112-12-15 02:19:00    65.0   
..          ...       ...       ...                  ...     ...   
748    11972365  21606206  31850358  2198-03-25 11:13:00     NaN   
749    11972365  21606206  31850358  2198-03-26 05:17:00     NaN   
750    14065397  25219971  39439699  2199-11-20 15:11:00    94.0   
751    14065397  25219971  39439699  2199-11-20 18:03:00    94.0   
752    17417573  24710404  36358936  2208-08-12 13:55:00    92.0   

    valueuom_220224  220227 valueuom_220227  220228 valueuom_220228  ...  \
0              mmHg    97.0               %     8.4            g/dl  ...   
1              mmHg   

In [37]:
import pandas as pd

# 读取drug.csv文件
data = pd.read_csv('./wenjian/drug.csv')

# 将starttime列数据转换为日期格式
data['date'] = pd.to_datetime(data['starttime'], format='%Y/%m/%d %H:%M').dt.date

# 对同一个 subject_id 下的 date 列数据种类进行计数
date_counts = data.groupby('subject_id')['date'].nunique()

# 将计数结果保存为day列数据
data['day'] = data['subject_id'].map(date_counts)

# 删除 date 列
data = data.drop(columns=['date'])

# 保存结果到新的文件 S9.csv
data.to_csv('S9.csv', index=False)

print(data)

     subject_id    ordercategoryname         starttime  value  \
0      16504173  08-Antibiotics (IV)   2110/6/19 22:00    4.3   
1      16504173  08-Antibiotics (IV)   2110/6/20 10:10    7.1   
2      15795343  08-Antibiotics (IV)   2112/3/12 10:15   12.8   
3      19122448  08-Antibiotics (IV)  2112/12/11 18:37    0.6   
4      19122448  08-Antibiotics (IV)  2112/12/11 18:37    0.4   
..          ...                  ...               ...    ...   
748    11972365  08-Antibiotics (IV)   2198/3/23 20:31    1.2   
749    11972365  08-Antibiotics (IV)   2198/3/23 20:31    1.3   
750    14065397  08-Antibiotics (IV)  2199/11/20 10:28    1.8   
751    14065397  08-Antibiotics (IV)  2199/11/20 16:20    1.8   
752    17417573  08-Antibiotics (IV)   2208/8/11 10:36    3.0   

            charttime  amount amountuom  totalamount totalamountuom  day  
0      2110/6/20 9:17     1.0      dose          100             ml    2  
1     2110/6/20 12:20     1.0      dose          100             ml  

In [39]:
import pandas as pd

# 读取 S9.csv 文件
s9_df = pd.read_csv('./wenjian/S9.csv')

# 将 charttime 和 starttime 列转换为 datetime 类型
s9_df['charttime'] = pd.to_datetime(s9_df['charttime'])
s9_df['starttime'] = pd.to_datetime(s9_df['starttime'])

# 计算 charttime 与 starttime 之间的时间差，并将结果转换为小时，四舍五入保留一位小数
s9_df['duration'] = ((s9_df['charttime'] - s9_df['starttime']).dt.total_seconds() / 3600).round(1)

print(s9_df)

# 保存处理后的 DataFrame 到新的 CSV 文件
s9_df.to_csv('S10.csv', index=False)


     subject_id    ordercategoryname           starttime  value  \
0      16504173  08-Antibiotics (IV) 2110-06-19 22:00:00    4.3   
1      16504173  08-Antibiotics (IV) 2110-06-20 10:10:00    7.1   
2      15795343  08-Antibiotics (IV) 2112-03-12 10:15:00   12.8   
3      19122448  08-Antibiotics (IV) 2112-12-11 18:37:00    0.6   
4      19122448  08-Antibiotics (IV) 2112-12-11 18:37:00    0.4   
..          ...                  ...                 ...    ...   
748    11972365  08-Antibiotics (IV) 2198-03-23 20:31:00    1.2   
749    11972365  08-Antibiotics (IV) 2198-03-23 20:31:00    1.3   
750    14065397  08-Antibiotics (IV) 2199-11-20 10:28:00    1.8   
751    14065397  08-Antibiotics (IV) 2199-11-20 16:20:00    1.8   
752    17417573  08-Antibiotics (IV) 2208-08-11 10:36:00    3.0   

              charttime  amount amountuom  totalamount totalamountuom  day  \
0   2110-06-20 09:17:00     1.0      dose          100             ml    2   
1   2110-06-20 12:20:00     1.0      do

In [31]:
import pandas as pd
import numpy as np
# 读取 CSV 文件
df = pd.read_csv('./wenjian/S10.csv')

# 将 'starttime' 和 'charttime' 列转换为日期时间格式
df['starttime'] = pd.to_datetime(df['starttime'])

# 根据 'subject_id' 进行分组
grouped = df.groupby('subject_id')

# 定义计算 interval 的函数
def calculate_interval(group):
    if group['starttime'].duplicated().any():  # 如果 starttime 有重复值
        group['interval'] = group['duration']
    else:
        group['interval'] = 0
    return group

# 对每个 subject_id 分组应用计算 interval 的函数
df = grouped.apply(calculate_interval)

# 新创建的 'duration_diff'，包含了每个测量浓度与前一个测量浓度之间的时间间隔
# 使用 NumPy 的 np.append 函数，在计算得到的时间间隔数组前面添加了一个0
# 后续的第2、3、、、行内容减去第1、2、、、行内容
df['duration_diff']=np.append(0,df.duration[1:].values-df.duration[:-1].values)

# 把负值替换为0
df['duration_diff'] = np.where(df['duration_diff'] < 0, 0, df['duration_diff'])

# 将 'interval' 列中非零数据填充为 'duration_diff' 列相应数据
df['interval'] = np.where(df['interval'] != 0, df['duration_diff'], df['interval'])

print(df)

df.to_csv('sss.csv')

                subject_id    ordercategoryname           starttime  value  \
subject_id                                                                   
10035168   365    10035168  08-Antibiotics (IV) 2145-12-26 09:42:00    2.7   
           366    10035168  08-Antibiotics (IV) 2145-12-27 12:20:00    3.9   
10036086   744    10036086  08-Antibiotics (IV) 2196-05-24 18:10:00    0.7   
           745    10036086  08-Antibiotics (IV) 2196-05-25 18:00:00    0.5   
           746    10036086  08-Antibiotics (IV) 2196-05-27 05:00:00    1.0   
...                    ...                  ...                 ...    ...   
19970491   181    19970491  08-Antibiotics (IV) 2129-07-15 08:00:00    2.5   
           182    19970491  08-Antibiotics (IV) 2129-07-15 08:00:00    0.7   
           183    19970491  08-Antibiotics (IV) 2129-07-17 20:00:00    2.0   
           184    19970491  08-Antibiotics (IV) 2129-07-17 20:00:00    1.0   
19985293   691    19985293  08-Antibiotics (IV) 2184-08-20 05:41

In [32]:
import pandas as pd

# 读取 CSV 文件
df = pd.read_csv('sss.csv')

# 将 starttime 和 charttime 转换为日期时间类型
df['starttime'] = pd.to_datetime(df['starttime'])
df['charttime'] = pd.to_datetime(df['charttime'])

# 按照 subject_id 分组，并找到每个 subject_id 的重复 starttime 对应的最小 charttime
min_charttime = df.groupby(['subject_id', 'starttime'])['charttime'].min().reset_index()

# 将找到的最小 charttime 对应的 interval 列数据赋值为 0
for index, row in min_charttime.iterrows():
    subject_id = row['subject_id']
    starttime = row['starttime']
    min_charttime = row['charttime']
    df.loc[(df['subject_id'] == subject_id) & (df['starttime'] == starttime) & (df['charttime'] == min_charttime), 'interval'] = 0

# 删除 duration_diff 列
df.drop(columns=['duration_diff'], inplace=True)

# 将 interval 列数据四舍五入保留一位小数
df['interval'] = df['interval'].round(1)

# 输出更新后的 DataFrame
print(df)

df.to_csv('S11.csv')

     subject_id  Unnamed: 1  subject_id.1    ordercategoryname  \
0      10035168         365      10035168  08-Antibiotics (IV)   
1      10035168         366      10035168  08-Antibiotics (IV)   
2      10036086         744      10036086  08-Antibiotics (IV)   
3      10036086         745      10036086  08-Antibiotics (IV)   
4      10036086         746      10036086  08-Antibiotics (IV)   
..          ...         ...           ...                  ...   
748    19970491         181      19970491  08-Antibiotics (IV)   
749    19970491         182      19970491  08-Antibiotics (IV)   
750    19970491         183      19970491  08-Antibiotics (IV)   
751    19970491         184      19970491  08-Antibiotics (IV)   
752    19985293         691      19985293  08-Antibiotics (IV)   

              starttime  value           charttime      amount amountuom  \
0   2145-12-26 09:42:00    2.7 2145-12-26 17:29:00  120.000005        mg   
1   2145-12-27 12:20:00    3.9 2145-12-27 19:34:00  100

In [34]:
import pandas as pd

# 读取 CSV 文件
df = pd.read_csv('./wenjian/S11.csv')

# 将 starttime 列转换为日期时间类型
df['starttime'] = pd.to_datetime(df['starttime'])

# 计算每个 subject_id 对应的 starttime 数据的种类数量
starttime_count = df.groupby('subject_id')['starttime'].nunique()

# 将计算结果填充到一个新列中
df['sample'] = df['subject_id'].map(starttime_count)

# 将更新后的 DataFrame 保存到 S11.csv 文件中
df.to_csv('S12.csv', index=False)

# 打印更新后的 DataFrame
print(df)

     subject_id    ordercategoryname           starttime  value  \
0      10035168  08-Antibiotics (IV) 2145-12-26 09:42:00    2.7   
1      10035168  08-Antibiotics (IV) 2145-12-27 12:20:00    3.9   
2      10036086  08-Antibiotics (IV) 2196-05-24 18:10:00    0.7   
3      10036086  08-Antibiotics (IV) 2196-05-25 18:00:00    0.5   
4      10036086  08-Antibiotics (IV) 2196-05-27 05:00:00    1.0   
..          ...                  ...                 ...    ...   
748    19970491  08-Antibiotics (IV) 2129-07-15 08:00:00    2.5   
749    19970491  08-Antibiotics (IV) 2129-07-15 08:00:00    0.7   
750    19970491  08-Antibiotics (IV) 2129-07-17 20:00:00    2.0   
751    19970491  08-Antibiotics (IV) 2129-07-17 20:00:00    1.0   
752    19985293  08-Antibiotics (IV) 2184-08-20 05:41:00    7.9   

               charttime      amount amountuom  totalamount totalamountuom  \
0    2145-12-26 17:29:00  120.000005        mg          100             ml   
1    2145-12-27 19:34:00  100.000001   

In [30]:
import pandas as pd

# 读取 label.csv 文件，保存itemid与其对应的label
label_data = pd.read_csv('./wenjian/lable.csv')

# 读取 part2.csv 文件
part2_data = pd.read_csv('./wenjian/part2.csv')

# 将 label.csv 文件中的数据转换为字典，key 为 itemid，value 为 label
label_dict = dict(zip(label_data['itemid'], label_data['label']))

# 将 part2.csv 文件的列名转换为整数类型，并根据 label_dict 字典修改列名
part2_data.columns = [int(col) if col.isdigit() else col for col in part2_data.columns]
part2_data.columns = [label_dict[col] if col in label_dict else col for col in part2_data.columns]

# 将修改后的数据保存到新的 CSV 文件中
part2_data.to_csv('part2.csv', index=False)
print(part2_data)

     subject_id  gender  age     race  weight  height  creatinine  BUN  \
0      10035168       0   56    WHITE    74.0   170.0         0.8   30   
1      10035168       0   56    WHITE    74.0   170.0         1.0   38   
2      10036086       1   57    WHITE    97.9   173.0         2.7   46   
3      10036086       1   57    WHITE    77.5   173.0         2.3   39   
4      10036086       1   57    WHITE    77.5   173.0         1.9   29   
..          ...     ...  ...      ...     ...     ...         ...  ...   
748    19970491       1   55    WHITE    78.4     NaN         0.8   34   
749    19970491       1   55    WHITE    78.4     NaN         1.0   36   
750    19970491       1   55    WHITE    58.3     NaN         1.1   42   
751    19970491       1   55    WHITE    58.3     NaN         1.1   43   
752    19985293       0   81  UNKNOWN   116.0   150.0         1.0   19   

     albumin    ALT  ...  CentralVenousO2%Sat  \
0        2.9   35.0  ...                  NaN   
1        2.9 

In [34]:
import pandas as pd

# 读取CSV文件
df = pd.read_csv('./wenjian/S12.csv')

# 添加新列，这里以0或空字符串初始化
df['dose_val_rx'] = ''
df['dose_unit_rx'] = ''
df['doses_per_24_hrs'] = ''
df['route'] = ''

# 保存到新的CSV文件
df.to_csv('S12_updated.csv', index=False)

pinci = pd.read_csv("./wenjian/pinci.csv")
s12 = pd.read_csv("S12_updated.csv")

i = 2
#j = 1

# 转换时间列为 pandas 的 datetime 类型
pinci['starttime1'] = pd.to_datetime(pinci['starttime1'])
pinci['stoptime'] = pd.to_datetime(pinci['stoptime'])
s12['starttime'] = pd.to_datetime(s12['starttime'])

# 新建一个空列表
indices_list = []

# 迭代 S12_updated.csv 的每一行
for idx, row in s12.iterrows():
    # 在 pinci.csv 中找到相同 hadm_id 的行并检查时间区间
    mask = (pinci['hadm_id'] == row['hadm_id']) & \
           (pinci['starttime1'] <= row['starttime']) & \
           (pinci['stoptime'] >= row['starttime'])
    matching_rows = pinci[mask]
    print('第' + str(i) + '行有' + str(len(matching_rows)) + '个符合数据')
    
    # if len(matching_rows) > 2:
    #     print(str(i))
    if len(matching_rows) == 2:
    #     # 将符合条件的索引添加到列表中
        indices_list.append(str(i))
    # 将列表写入 CSV 文件
    with open('output.csv', 'w') as f:
        for index in indices_list:
            f.write("%s\n" % index)
    
    i = i + 1
    
    if (len(matching_rows) == 1 ):
        match = matching_rows.iloc[0]
        # 从 pinci 复制数据到 s12
        s12.at[idx, 'dose_val_rx'] = match['dose_val_rx']
        s12.at[idx, 'dose_unit_rx'] = match['dose_unit_rx']
        s12.at[idx, 'doses_per_24_hrs'] = match['doses_per_24_hrs']
        s12.at[idx, 'route'] = match['route']
    
# 保存更新后的文件
s12.to_csv("S13.csv", index=False)
print(s12)

第2行有2个符合数据
第3行有1个符合数据
第4行有1个符合数据
第5行有1个符合数据
第6行有1个符合数据
第7行有1个符合数据
第8行有1个符合数据
第9行有1个符合数据
第10行有0个符合数据
第11行有1个符合数据
第12行有1个符合数据
第13行有1个符合数据
第14行有1个符合数据
第15行有1个符合数据
第16行有0个符合数据
第17行有0个符合数据
第18行有0个符合数据
第19行有0个符合数据
第20行有1个符合数据
第21行有1个符合数据
第22行有1个符合数据
第23行有1个符合数据
第24行有1个符合数据
第25行有1个符合数据
第26行有1个符合数据
第27行有1个符合数据
第28行有1个符合数据
第29行有0个符合数据
第30行有1个符合数据
第31行有1个符合数据
第32行有1个符合数据
第33行有1个符合数据
第34行有1个符合数据
第35行有1个符合数据
第36行有1个符合数据
第37行有1个符合数据
第38行有1个符合数据
第39行有0个符合数据
第40行有1个符合数据
第41行有1个符合数据
第42行有1个符合数据
第43行有1个符合数据
第44行有1个符合数据
第45行有1个符合数据
第46行有1个符合数据
第47行有1个符合数据
第48行有0个符合数据
第49行有1个符合数据
第50行有1个符合数据
第51行有1个符合数据
第52行有1个符合数据
第53行有1个符合数据
第54行有1个符合数据
第55行有1个符合数据
第56行有1个符合数据
第57行有1个符合数据
第58行有1个符合数据
第59行有1个符合数据
第60行有1个符合数据
第61行有1个符合数据
第62行有1个符合数据
第63行有2个符合数据
第64行有3个符合数据
第65行有1个符合数据
第66行有1个符合数据
第67行有1个符合数据
第68行有1个符合数据
第69行有1个符合数据
第70行有1个符合数据
第71行有1个符合数据
第72行有1个符合数据
第73行有1个符合数据
第74行有1个符合数据
第75行有1个符合数据
第76行有1个符合数据
第77行有1个符合数据
第78行有1个符合数据
第79行有1个符合数据
第80行有1个符合数据
第81行有1个符合数据
第82行有1个符合数据
第83行有1个符合数据
第84行有1个符合数据
第85行有1个符合数据


C:\Windows\Temp\ipykernel_7688\2009321584.py:54: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'mg' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  s12.at[idx, 'dose_unit_rx'] = match['dose_unit_rx']
C:\Windows\Temp\ipykernel_7688\2009321584.py:56: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'IV' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  s12.at[idx, 'route'] = match['route']


第195行有1个符合数据
第196行有1个符合数据
第197行有1个符合数据
第198行有0个符合数据
第199行有1个符合数据
第200行有1个符合数据
第201行有1个符合数据
第202行有1个符合数据
第203行有1个符合数据
第204行有1个符合数据
第205行有1个符合数据
第206行有1个符合数据
第207行有1个符合数据
第208行有1个符合数据
第209行有1个符合数据
第210行有1个符合数据
第211行有1个符合数据
第212行有1个符合数据
第213行有1个符合数据
第214行有1个符合数据
第215行有1个符合数据
第216行有1个符合数据
第217行有1个符合数据
第218行有0个符合数据
第219行有0个符合数据
第220行有0个符合数据
第221行有0个符合数据
第222行有0个符合数据
第223行有0个符合数据
第224行有0个符合数据
第225行有1个符合数据
第226行有1个符合数据
第227行有1个符合数据
第228行有1个符合数据
第229行有1个符合数据
第230行有1个符合数据
第231行有1个符合数据
第232行有1个符合数据
第233行有1个符合数据
第234行有2个符合数据
第235行有1个符合数据
第236行有1个符合数据
第237行有1个符合数据
第238行有1个符合数据
第239行有1个符合数据
第240行有1个符合数据
第241行有1个符合数据
第242行有1个符合数据
第243行有1个符合数据
第244行有1个符合数据
第245行有1个符合数据
第246行有1个符合数据
第247行有1个符合数据
第248行有1个符合数据
第249行有1个符合数据
第250行有1个符合数据
第251行有1个符合数据
第252行有2个符合数据
第253行有2个符合数据
第254行有1个符合数据
第255行有1个符合数据
第256行有1个符合数据
第257行有1个符合数据
第258行有1个符合数据
第259行有1个符合数据
第260行有1个符合数据
第261行有1个符合数据
第262行有1个符合数据
第263行有1个符合数据
第264行有1个符合数据
第265行有1个符合数据
第266行有1个符合数据
第267行有1个符合数据
第268行有1个符合数据
第269行有1个符合数据
第270行有1个符合数据
第271行有1个符合数据

In [35]:
import pandas as pd

# 读取drug.csv文件
data = pd.read_csv('./wenjian/S13.csv')

# 将starttime列数据转换为日期格式
data['date'] = pd.to_datetime(data['starttime'], format='%Y/%m/%d %H:%M').dt.date

# 对同一个 subject_id 下的 date 列数据种类进行计数
date_counts = data.groupby('subject_id')['date'].nunique()

# 将计数结果保存为day列数据
data['day'] = data['subject_id'].map(date_counts)

# 删除 date 列
data = data.drop(columns=['date'])

# 保存结果到新的文件 S14.csv
data.to_csv('S14.csv', index=False)

print(data)


     subject_id   hadm_id         starttime         charttime  gender  age  \
0      10035168  25449821   2145/12/26 9:42  2145/12/26 17:29       0   56   
1      10035168  25449821  2145/12/27 12:20  2145/12/27 19:34       0   56   
2      10036086  28728587   2196/5/24 18:10   2196/5/25 17:38       1   57   
3      10036086  28728587   2196/5/25 18:00   2196/5/26 15:53       1   57   
4      10036086  28728587    2196/5/27 5:00   2196/5/27 15:03       1   57   
..          ...       ...               ...               ...     ...  ...   
662    19929286  24868766  2193/11/12 19:49  2193/11/13 19:08       0   64   
663    19965582  22946607     2186/9/2 5:01     2186/9/2 8:21       1   55   
664    19970491  25338284   2129/7/17 20:00    2129/7/19 6:00       1   55   
665    19970491  25338284   2129/7/17 20:00   2129/7/19 19:26       1   55   
666    19985293  21731208    2184/8/20 5:41   2184/8/20 13:41       0   81   

        race  weight  height  dose_val_rx  ...  \
0      WHITE 

In [36]:
import pandas as pd

# 读取 S14.csv 文件
s9_df = pd.read_csv('./wenjian/S14.csv')

# 将 charttime 和 starttime 列转换为 datetime 类型
s9_df['charttime'] = pd.to_datetime(s9_df['charttime'])
s9_df['starttime'] = pd.to_datetime(s9_df['starttime'])

# 计算 charttime 与 starttime 之间的时间差，并将结果转换为小时，四舍五入保留一位小数
s9_df['duration'] = ((s9_df['charttime'] - s9_df['starttime']).dt.total_seconds() / 3600).round(1)

print(s9_df)

# 保存处理后的 DataFrame 到新的 CSV 文件
s9_df.to_csv('S15.csv', index=False)


     subject_id   hadm_id           starttime           charttime  gender  \
0      10035168  25449821 2145-12-26 09:42:00 2145-12-26 17:29:00       0   
1      10035168  25449821 2145-12-27 12:20:00 2145-12-27 19:34:00       0   
2      10036086  28728587 2196-05-24 18:10:00 2196-05-25 17:38:00       1   
3      10036086  28728587 2196-05-25 18:00:00 2196-05-26 15:53:00       1   
4      10036086  28728587 2196-05-27 05:00:00 2196-05-27 15:03:00       1   
..          ...       ...                 ...                 ...     ...   
662    19929286  24868766 2193-11-12 19:49:00 2193-11-13 19:08:00       0   
663    19965582  22946607 2186-09-02 05:01:00 2186-09-02 08:21:00       1   
664    19970491  25338284 2129-07-17 20:00:00 2129-07-19 06:00:00       1   
665    19970491  25338284 2129-07-17 20:00:00 2129-07-19 19:26:00       1   
666    19985293  21731208 2184-08-20 05:41:00 2184-08-20 13:41:00       0   

     age     race  weight  height  dose_val_rx  ... AbsoluteCount-Neuts  \


In [37]:
import pandas as pd
import numpy as np

# 读取 CSV 文件
df = pd.read_csv('./wenjian/S15.csv')

# 将 'starttime' 和 'charttime' 列转换为日期时间格式
df['starttime'] = pd.to_datetime(df['starttime'])

# 根据 'subject_id' 进行分组
grouped = df.groupby('subject_id')

# 定义计算 interval 的函数
def calculate_interval(group):
    if group['starttime'].duplicated().any():  # 如果 starttime 有重复值
        group['interval'] = group['duration']
    else:
        group['interval'] = 0
    return group

# 对每个 subject_id 分组应用计算 interval 的函数
df = grouped.apply(calculate_interval)

# 新创建的 'duration_diff'，包含了每个测量浓度与前一个测量浓度之间的时间间隔
# 使用 NumPy 的 np.append 函数，在计算得到的时间间隔数组前面添加了一个0
# 后续的第2、3、、、行内容减去第1、2、、、行内容
df['duration_diff']=np.append(0,df.duration[1:].values-df.duration[:-1].values)

# 把负值替换为0
df['duration_diff'] = np.where(df['duration_diff'] < 0, 0, df['duration_diff'])

# 将 'interval' 列中非零数据填充为 'duration_diff' 列相应数据
df['interval'] = np.where(df['interval'] != 0, df['duration_diff'], df['interval'])

print(df)

df.to_csv('sss.csv')


                subject_id   hadm_id           starttime            charttime  \
subject_id                                                                      
10035168   0      10035168  25449821 2145-12-26 09:42:00  2145-12-26 17:29:00   
           1      10035168  25449821 2145-12-27 12:20:00  2145-12-27 19:34:00   
10036086   2      10036086  28728587 2196-05-24 18:10:00  2196-05-25 17:38:00   
           3      10036086  28728587 2196-05-25 18:00:00  2196-05-26 15:53:00   
           4      10036086  28728587 2196-05-27 05:00:00  2196-05-27 15:03:00   
...                    ...       ...                 ...                  ...   
19929286   662    19929286  24868766 2193-11-12 19:49:00  2193-11-13 19:08:00   
19965582   663    19965582  22946607 2186-09-02 05:01:00  2186-09-02 08:21:00   
19970491   664    19970491  25338284 2129-07-17 20:00:00  2129-07-19 06:00:00   
           665    19970491  25338284 2129-07-17 20:00:00  2129-07-19 19:26:00   
19985293   666    19985293  

In [39]:
import pandas as pd

# 读取 CSV 文件
df = pd.read_csv('./wenjian/sss.csv')

# 将 starttime 和 charttime 转换为日期时间类型
df['starttime'] = pd.to_datetime(df['starttime'])
df['charttime'] = pd.to_datetime(df['charttime'])

# 按照 subject_id 分组，并找到每个 subject_id 的重复 starttime 对应的最小 charttime
min_charttime = df.groupby(['subject_id', 'starttime'])['charttime'].min().reset_index()

# 将找到的最小 charttime 对应的 interval 列数据赋值为 0
for index, row in min_charttime.iterrows():
    subject_id = row['subject_id']
    starttime = row['starttime']
    min_charttime = row['charttime']
    df.loc[(df['subject_id'] == subject_id) & (df['starttime'] == starttime) & (df['charttime'] == min_charttime), 'interval'] = 0

# 删除 duration_diff 列
df.drop(columns=['duration_diff'], inplace=True)

#将interval列数据四舍五入保留一位小数
df['interval']=df['interval'].round(1)

# 输出更新后的 DataFrame
print(df)

df.to_csv('S16.csv')


     subject_id   hadm_id           starttime           charttime  gender  \
0      10035168  25449821 2145-12-26 09:42:00 2145-12-26 17:29:00       0   
1      10035168  25449821 2145-12-27 12:20:00 2145-12-27 19:34:00       0   
2      10036086  28728587 2196-05-24 18:10:00 2196-05-25 17:38:00       1   
3      10036086  28728587 2196-05-25 18:00:00 2196-05-26 15:53:00       1   
4      10036086  28728587 2196-05-27 05:00:00 2196-05-27 15:03:00       1   
..          ...       ...                 ...                 ...     ...   
662    19929286  24868766 2193-11-12 19:49:00 2193-11-13 19:08:00       0   
663    19965582  22946607 2186-09-02 05:01:00 2186-09-02 08:21:00       1   
664    19970491  25338284 2129-07-17 20:00:00 2129-07-19 06:00:00       1   
665    19970491  25338284 2129-07-17 20:00:00 2129-07-19 19:26:00       1   
666    19985293  21731208 2184-08-20 05:41:00 2184-08-20 13:41:00       0   

     age     race  weight  height  dose_val_rx  ... AbsoluteCount-Lymphs  \

In [40]:
import pandas as pd

# 读取 CSV 文件
df = pd.read_csv('./wenjian/S16.csv')

# 将 starttime 列转换为日期时间类型
df['starttime'] = pd.to_datetime(df['starttime'])

# 计算每个 subject_id 对应的 starttime 数据的种类数量
starttime_count = df.groupby('subject_id')['starttime'].nunique()

# 将计算结果填充到一个新列中
df['sample'] = df['subject_id'].map(starttime_count)

# 将更新后的 DataFrame 保存到 S11.csv 文件中
df.to_csv('S17.csv', index=False)

# 打印更新后的 DataFrame
print(df)


     Unnamed: 0  subject_id   hadm_id           starttime  \
0             0    10035168  25449821 2145-12-26 09:42:00   
1             1    10035168  25449821 2145-12-27 12:20:00   
2             2    10036086  28728587 2196-05-24 18:10:00   
3             3    10036086  28728587 2196-05-25 18:00:00   
4             4    10036086  28728587 2196-05-27 05:00:00   
..          ...         ...       ...                 ...   
662         662    19929286  24868766 2193-11-12 19:49:00   
663         663    19965582  22946607 2186-09-02 05:01:00   
664         664    19970491  25338284 2129-07-17 20:00:00   
665         665    19970491  25338284 2129-07-17 20:00:00   
666         666    19985293  21731208 2184-08-20 05:41:00   

               charttime  gender  age     race  weight  height  ...  \
0    2145-12-26 17:29:00       0   56    WHITE    74.0   170.0  ...   
1    2145-12-27 19:34:00       0   56    WHITE    74.0   170.0  ...   
2    2196-05-25 17:38:00       1   57    WHITE    97.9

In [45]:
import pandas as pd

# 读取 XIHE.csv 文件
xihe_data = pd.read_csv('XIHE.csv')

# 读取 S17.csv 文件
s17_data = pd.read_csv('./wenjian/S17.csv')

# 根据 hadm_id 列合并 XIHE.csv 和 S17.csv 文件
merged_data = pd.merge(s17_data, xihe_data[['hadm_id', 'value']], on='hadm_id', how='left')

# 将 value 列数据填充到 S17.csv 文件后
s17_data['value'] = merged_data['value']

# 将修改后的数据保存到新的 CSV 文件中
s17_data.to_csv('S18.csv', index=False)
print(s17_data)

     subject_id   hadm_id            starttime            charttime  gender  \
0      10035168  25449821  2145-12-26 09:42:00  2145-12-26 17:29:00       0   
1      10035168  25449821  2145-12-27 12:20:00  2145-12-27 19:34:00       0   
2      10036086  28728587  2196-05-24 18:10:00  2196-05-25 17:38:00       1   
3      10036086  28728587  2196-05-25 18:00:00  2196-05-26 15:53:00       1   
4      10036086  28728587  2196-05-27 05:00:00  2196-05-27 15:03:00       1   
..          ...       ...                  ...                  ...     ...   
662    19929286  24868766  2193-11-12 19:49:00  2193-11-13 19:08:00       0   
663    19965582  22946607  2186-09-02 05:01:00  2186-09-02 08:21:00       1   
664    19970491  25338284  2129-07-17 20:00:00  2129-07-19 06:00:00       1   
665    19970491  25338284  2129-07-17 20:00:00  2129-07-19 19:26:00       1   
666    19985293  21731208  2184-08-20 05:41:00  2184-08-20 13:41:00       0   

     age     race  weight  height  dose_val_rx  ...

In [49]:
import pandas as pd

# 读取 CSV 文件
data = pd.read_csv('./wenjian/S18.csv')

# 创建一个字典来存储每个不同的 "subject_id" 对应的序号
subject_id_mapping = {}
new_subject_ids = []

# 遍历数据集，为每个不同的 "subject_id" 分配一个唯一的序号
current_id = 1
for subject_id in data['subject_id']:
    if subject_id not in subject_id_mapping:
        subject_id_mapping[subject_id] = current_id
        current_id += 1
    new_subject_ids.append(subject_id_mapping[subject_id])

# 将 "subject_id" 列替换为新的序号
data['subject_id'] = new_subject_ids

# 保存修改后的数据到新文件
data.to_csv('S18.csv', index=False)
print(data)

     subject_id  value  gender  age     race  weight  height  drinking  \
0             1    2.7       0   56    WHITE    74.0   170.0       0.0   
1             1    3.9       0   56    WHITE    74.0   170.0       0.0   
2             2    0.7       1   57    WHITE    97.9   173.0       0.0   
3             2    0.5       1   57    WHITE    77.5   173.0       0.0   
4             2    1.0       1   57    WHITE    77.5   173.0       0.0   
..          ...    ...     ...  ...      ...     ...     ...       ...   
662         259    1.0       0   64    WHITE   113.0     NaN       0.0   
663         260    3.7       1   55    WHITE    58.1   178.0       0.0   
664         261    2.0       1   55    WHITE    58.3     NaN       0.0   
665         261    1.0       1   55    WHITE    58.3     NaN       0.0   
666         262    7.9       0   81  UNKNOWN   116.0   150.0       0.0   

     dose_val_rx  doses_per_24_hrs  ... CentralVenousO2%Sat  \
0            120                 3  ...         